# Detector multicandidato de pastillas en camara

In [7]:
import csv
import glob
import os
import time
import cv2
import numpy as np

PILLS_FOLDER = 'pills'
PILLBOX_CSV = 'Pillbox.csv'
DISPLAY_SIZE = 720
CANVAS_SIZE = 96
CAMERA_INDEX = 1
WORK_AREA_SCALE = 0.62
MASK_MEMORY = 0.38
BOX_BLEND = 0.82
MIN_COMPONENT_AREA = 2200
MAX_COMPONENT_AREA_RATIO = 0.28
MATCH_ACCEPT = 0.62
MATCH_MARGIN = 0.12
ROUND_CLOSE_KERNEL = 23
NEUTRAL_SAT_LIMIT = 42
NEUTRAL_CHROMA_LIMIT = 18
BACKGROUND_REJECT_RATIO = 0.72
TRACK_RADIUS = 65
TRACK_MEMORY = 3
MAX_DETECTIONS = 2
PRINT_MIN_SAT = 8.0
PRINT_CHROMA_DELTA = 5.0
PRINT_LOCAL_CONTRAST = 7.0
PRINT_LOW_SAT_THRESHOLD = 46.0
PRINT_ACCEPT_DELTA = 0.03
PRINT_BICOLOR_MIN_RATIO = 0.14

catalog = [
    {
        'reference': '60429-203_M_LH3',
        'family': 'ROUND_SOLID',
        'pillbox_id': '37642',
        'product_code': '60429-203',
        'splimage': '60429-203_M_LH3',
        'pillbox_size': '9',
        'pillbox_mark': 'LH3;M',
        'pillbox_color': 'GREEN',
        'pillbox_shape': 'ROUND',
        'medicine_name': 'Lisinopril and Hydrochlorothiazide'
    },
    {
        'reference': '317220267',
        'family': 'ROUND_SOLID',
        'pillbox_id': '14454',
        'product_code': '31722-267',
        'splimage': '317220267',
        'pillbox_size': '6',
        'pillbox_mark': '267;IG',
        'pillbox_color': 'BROWN',
        'pillbox_shape': 'ROUND',
        'medicine_name': 'Quinapril'
    },
    {
        'reference': '634810623',
        'family': 'ROUND_SOLID',
        'pillbox_id': '35466',
        'product_code': '63481-623',
        'splimage': '634810623',
        'pillbox_size': '11',
        'pillbox_mark': 'PERCOCET;5',
        'pillbox_color': 'BLUE',
        'pillbox_shape': 'ROUND',
        'medicine_name': 'PERCOCET'
    },
    {
        'reference': 'blackNyellow_capsule',
        'family': 'CAPSULE_BICOLOR',
        'pillbox_id': '29289',
        'product_code': '0172-2407',
        'splimage': '00172-2407-80_A60ED366',
        'pillbox_size': '22',
        'pillbox_mark': 'Z;2407',
        'pillbox_color': 'BLACK;YELLOW',
        'pillbox_shape': 'CAPSULE',
        'medicine_name': 'Tetracycline Hydrochloride'
    },
    {
        'reference': 'bloe_oval',
        'family': 'OBLONG_SOLID',
        'pillbox_id': '15248',
        'product_code': '0173-0933',
        'splimage': '00173-0933-08_2E1D970C',
        'pillbox_size': '18',
        'pillbox_mark': 'VALTREX;500;mg',
        'pillbox_color': 'BLUE',
        'pillbox_shape': 'OVAL',
        'medicine_name': 'VALTREX'
    },
    {
        'reference': 'brown_oval',
        'family': 'OBLONG_SOLID',
        'pillbox_id': '28299',
        'product_code': '68462-293',
        'splimage': '68462-0293-05_NLMIMAGE10_F03CF837',
        'pillbox_size': '13',
        'pillbox_mark': '293',
        'pillbox_color': 'BROWN',
        'pillbox_shape': 'OVAL',
        'medicine_name': 'verapamil hydrochloride'
    },
    {
        'reference': 'cream_capsule',
        'family': 'CAPSULE_BICOLOR',
        'pillbox_id': '30406',
        'product_code': '0093-7384',
        'splimage': '00093738456',
        'pillbox_size': '18',
        'pillbox_mark': '93;7384;93;7384',
        'pillbox_color': 'GRAY;BROWN',
        'pillbox_shape': 'CAPSULE',
        'medicine_name': 'Venlafaxine Hydrochloride'
    },
    {
        'reference': 'green_capsule',
        'family': 'CAPSULE_SOLID',
        'pillbox_id': '32477',
        'product_code': '0093-7170',
        'splimage': '00093-7170-06_NLMIMAGE10_C542E2B7',
        'pillbox_size': '23',
        'pillbox_mark': 'TEVA;7170',
        'pillbox_color': 'GREEN',
        'pillbox_shape': 'CAPSULE',
        'medicine_name': 'Celecoxib'
    },
    {
        'reference': 'green_oval',
        'family': 'OBLONG_SOLID',
        'pillbox_id': '66604',
        'product_code': '42494-309',
        'splimage': 'a3ea1ebb-2601-d8f2-e053-2a95a90a1216',
        'pillbox_size': '16',
        'pillbox_mark': 'AS;400',
        'pillbox_color': 'YELLOW',
        'pillbox_shape': 'OVAL',
        'medicine_name': 'Amiodarone HCl'
    },
    {
        'reference': 'orangeandblue',
        'family': 'CAPSULE_BICOLOR',
        'pillbox_id': '33990',
        'product_code': '51285-539',
        'splimage': '51285053902',
        'pillbox_size': '18',
        'pillbox_mark': 'OP;719',
        'pillbox_color': 'BLUE;ORANGE',
        'pillbox_shape': 'CAPSULE',
        'medicine_name': 'Surmontil'
    }
]

reference_paths = {}
for ref_path in sorted(glob.glob(os.path.join(PILLS_FOLDER, '*.*'))):
    ref_name = os.path.splitext(os.path.basename(ref_path))[0]
    reference_paths[ref_name] = ref_path

pillbox_rows = []
if os.path.exists(PILLBOX_CSV):
    with open(PILLBOX_CSV, newline='', encoding='utf-8') as handle:
        pillbox_rows = list(csv.DictReader(handle))

for item in catalog:
    item['image_path'] = reference_paths.get(item['reference'], '')
    exact_rows = []
    for row in pillbox_rows:
        same_id = row.get('ID', '') == item['pillbox_id']
        same_code = row.get('product_code', '') == item['product_code']
        same_image = (not item['splimage']) or (row.get('splimage', '') == item['splimage'])
        if same_id and same_code and same_image:
            exact_rows.append(row)
    if exact_rows:
        exact_row = exact_rows[0]
        item['pillbox_size'] = exact_row.get('splsize', '') or item['pillbox_size']
        item['pillbox_mark'] = exact_row.get('splimprint', '') or item['pillbox_mark']
        item['pillbox_color'] = exact_row.get('splcolor_text', '') or item['pillbox_color']
        item['pillbox_shape'] = exact_row.get('splshape_text', '') or item['pillbox_shape']
        item['medicine_name'] = exact_row.get('medicine_name', '') or item['medicine_name']
    item['mark_display'] = item['pillbox_mark'].replace(';', ' ')
    item['size_display'] = str(item['pillbox_size'])
    item['family_display'] = item['family'].replace('_', ' ')

print('Catalogo cargado desde pills/ y referenciado a Pillbox.csv:')
for item in catalog:
    path_state = 'OK' if item['image_path'] else 'FALTA IMAGEN'
    print(f"- {item['reference']:<22s} | {item['family_display']:<15s} | size={item['size_display']:<3s} | mark={item['mark_display']:<20s} | {path_state}")

Catalogo cargado desde pills/ y referenciado a Pillbox.csv:
- 60429-203_M_LH3        | ROUND SOLID     | size=9   | mark=LH3 M                | OK
- 317220267              | ROUND SOLID     | size=6   | mark=267 IG               | OK
- 634810623              | ROUND SOLID     | size=11  | mark=PERCOCET 5           | OK
- blackNyellow_capsule   | CAPSULE BICOLOR | size=22  | mark=Z 2407               | OK
- bloe_oval              | OBLONG SOLID    | size=18  | mark=VALTREX 500 mg       | OK
- brown_oval             | OBLONG SOLID    | size=13  | mark=293                  | OK
- cream_capsule          | CAPSULE BICOLOR | size=18  | mark=93 7384 93 7384      | OK
- green_capsule          | CAPSULE SOLID   | size=23  | mark=TEVA 7170            | OK
- green_oval             | OBLONG SOLID    | size=16  | mark=AS 400               | OK
- orangeandblue          | CAPSULE BICOLOR | size=18  | mark=OP 719               | OK


## Calibración

In [8]:
COLOR_MASKS_HSV = {
    'BLACK': [((0, 0, 0), (180, 255, 55))],
    'WHITE': [((0, 0, 185), (180, 30, 255))],
    'GRAY': [((0, 0, 70), (180, 55, 200))],
    'BROWN': [((5, 55, 40), (20, 255, 190))],
    'RED': [((0, 70, 70), (8, 255, 255)), ((170, 70, 70), (180, 255, 255))],
    'ORANGE': [((8, 75, 100), (22, 255, 255))],
    'YELLOW': [((22, 60, 110), (35, 255, 255))],
    'GREEN': [((35, 45, 60), (85, 255, 255))],
    'BLUE': [((85, 45, 50), (135, 255, 255))],
    'PINK': [((135, 45, 90), (170, 255, 255))]
}

def classify_colors_from_mask(image_hsv, object_mask, min_region_ratio=0.015, min_color_ratio=0.08):
    object_mask = (object_mask > 0).astype(np.uint8) * 255
    total_pixels = max(int(np.count_nonzero(object_mask)), 1)
    min_region_area = max(120, int(total_pixels * min_region_ratio))
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    color_counts = {}

    for color_name, hsv_ranges in COLOR_MASKS_HSV.items():
        color_mask = np.zeros(object_mask.shape, dtype=np.uint8)
        for lower, upper in hsv_ranges:
            lower_np = np.array(lower, dtype=np.uint8)
            upper_np = np.array(upper, dtype=np.uint8)
            color_mask = cv2.bitwise_or(color_mask, cv2.inRange(image_hsv, lower_np, upper_np))

        color_mask = cv2.bitwise_and(color_mask, object_mask)
        color_mask = cv2.morphologyEx(color_mask, cv2.MORPH_OPEN, kernel_open, iterations=1)
        color_mask = cv2.morphologyEx(color_mask, cv2.MORPH_CLOSE, kernel_close, iterations=1)
        contours, _ = cv2.findContours(color_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        kept_area = 0
        for contour in contours:
            contour_area = int(cv2.contourArea(contour))
            if contour_area < min_region_area:
                continue
            kept_area += contour_area

        if kept_area > 0:
            color_counts[color_name] = kept_area

    color_pairs = sorted(color_counts.items(), key=lambda pair: pair[1], reverse=True)
    dominant_colors = [pair[0] for pair in color_pairs[:3]]
    dominant_props = [float(pair[1] / total_pixels) for pair in color_pairs[:3]]
    if not dominant_colors:
        dominant_colors = ['UNKNOWN']
        dominant_props = [0.0]

    is_bicolor = len(dominant_colors) > 1 and dominant_props[0] >= min_color_ratio and dominant_props[1] >= min_color_ratio and dominant_colors[0] != dominant_colors[1]
    display_color = dominant_colors[0]
    if is_bicolor:
        display_color = dominant_colors[0] + '/' + dominant_colors[1]

    return {
        'color_counts': color_counts,
        'dominant_colors': dominant_colors,
        'dominant_props': dominant_props,
        'display_color': display_color,
        'is_bicolor': is_bicolor,
        'total_pixels': total_pixels
    }

def build_color_support_mask(image_hsv):
    support_mask = np.zeros(image_hsv.shape[:2], dtype=np.uint8)
    for hsv_ranges in COLOR_MASKS_HSV.values():
        color_mask = np.zeros(image_hsv.shape[:2], dtype=np.uint8)
        for lower, upper in hsv_ranges:
            lower_np = np.array(lower, dtype=np.uint8)
            upper_np = np.array(upper, dtype=np.uint8)
            color_mask = cv2.bitwise_or(color_mask, cv2.inRange(image_hsv, lower_np, upper_np))
        support_mask = cv2.bitwise_or(support_mask, color_mask)

    support_mask = cv2.morphologyEx(support_mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=1)
    support_mask = cv2.morphologyEx(support_mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)), iterations=1)
    return support_mask

reference_profiles = {}
family_buckets = {}

for item in catalog:
    reference_name = item['reference']
    image_path = item['image_path']
    if not image_path or not os.path.exists(image_path):
        print(f'- {reference_name}: sin imagen de referencia')
        continue

    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        print(f'- {reference_name}: no se pudo leer la imagen')
        continue

    scale = DISPLAY_SIZE / max(image_bgr.shape[:2])
    resized_w = max(1, int(image_bgr.shape[1] * scale))
    resized_h = max(1, int(image_bgr.shape[0] * scale))
    image_bgr = cv2.resize(image_bgr, (resized_w, resized_h), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC)

    image_lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    image_hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    image_gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    image_blur = cv2.GaussianBlur(image_gray, (5, 5), 0)
    image_eq = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image_blur)

    border = max(8, min(image_bgr.shape[:2]) // 18)
    border_lab = np.concatenate([
        image_lab[:border, :, :].reshape(-1, 3),
        image_lab[-border:, :, :].reshape(-1, 3),
        image_lab[:, :border, :].reshape(-1, 3),
        image_lab[:, -border:, :].reshape(-1, 3)
    ], axis=0).astype(np.float32)
    border_hsv = np.concatenate([
        image_hsv[:border, :, :].reshape(-1, 3),
        image_hsv[-border:, :, :].reshape(-1, 3),
        image_hsv[:, :border, :].reshape(-1, 3),
        image_hsv[:, -border:, :].reshape(-1, 3)
    ], axis=0).astype(np.float32)

    bg_lab = np.median(border_lab, axis=0)
    bg_lab_std = np.std(border_lab, axis=0)
    bg_sat = float(np.median(border_hsv[:, 1]))
    bg_val = float(np.median(border_hsv[:, 2]))
    bg_sat_std = float(np.std(border_hsv[:, 1]))

    lab_delta = np.abs(image_lab.astype(np.float32) - bg_lab.reshape(1, 1, 3))
    mask_lab = ((lab_delta[:, :, 0] > max(8.0, bg_lab_std[0] * 2.0)) | (lab_delta[:, :, 1] > max(10.0, bg_lab_std[1] * 1.9)) | (lab_delta[:, :, 2] > max(10.0, bg_lab_std[2] * 1.9))).astype(np.uint8) * 255
    sat_delta = np.abs(image_hsv[:, :, 1].astype(np.float32) - bg_sat)
    val_delta = np.abs(image_hsv[:, :, 2].astype(np.float32) - bg_val)
    mask_sv = ((sat_delta > max(18.0, bg_sat_std * 1.8)) | (val_delta > 24.0)).astype(np.uint8) * 255
    relaxed_color_gate = ((sat_delta > max(8.0, bg_sat_std * 1.0)) | (val_delta > 12.0) | (lab_delta[:, :, 1] > max(6.0, bg_lab_std[1] * 1.2)) | (lab_delta[:, :, 2] > max(6.0, bg_lab_std[2] * 1.2))).astype(np.uint8) * 255
    color_support_mask = build_color_support_mask(image_hsv)
    color_support_mask = cv2.bitwise_and(color_support_mask, relaxed_color_gate)
    mask_fg = cv2.max(cv2.max(mask_lab, mask_sv), color_support_mask)
    mask_fg = cv2.morphologyEx(mask_fg, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)), iterations=2)
    mask_fg = cv2.morphologyEx(mask_fg, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=1)
    mask_fg = cv2.medianBlur(mask_fg, 5)

    contours, _ = cv2.findContours(mask_fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    selected_contour = None
    image_area = float(image_bgr.shape[0] * image_bgr.shape[1])
    for contour in contours:
        contour_area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)
        fills_frame = w > image_bgr.shape[1] * 0.96 and h > image_bgr.shape[0] * 0.96
        if contour_area < image_area * 0.01:
            continue
        if contour_area > image_area * 0.92:
            continue
        if fills_frame:
            continue
        selected_contour = contour
        break

    if selected_contour is None:
        _, fallback_mask = cv2.threshold(image_eq, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        fallback_mask = cv2.morphologyEx(fallback_mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)), iterations=2)
        contours, _ = cv2.findContours(fallback_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contours = sorted(contours, key=cv2.contourArea, reverse=True)
        for contour in contours:
            contour_area = cv2.contourArea(contour)
            if contour_area >= image_area * 0.01 and contour_area <= image_area * 0.92:
                selected_contour = contour
                mask_fg = fallback_mask
                break

    if selected_contour is None:
        # Muchas referencias traen dos vistas de la pastilla sobre un fondo gris uniforme.
        # En esos casos, segmentamos por distancia al color del fondo y elegimos una vista valida.
        bg_distance = np.linalg.norm(image_lab.astype(np.float32) - bg_lab.reshape(1, 1, 3), axis=2)
        for distance_threshold in [20.0, 24.0, 28.0, 32.0]:
            distance_mask = (bg_distance > distance_threshold).astype(np.uint8) * 255
            distance_mask = cv2.morphologyEx(distance_mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=1)
            distance_mask = cv2.morphologyEx(distance_mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)), iterations=2)
            contours, _ = cv2.findContours(distance_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            contours = sorted(contours, key=cv2.contourArea, reverse=True)
            for contour in contours:
                contour_area = cv2.contourArea(contour)
                x, y, w, h = cv2.boundingRect(contour)
                fills_frame = w > image_bgr.shape[1] * 0.96 and h > image_bgr.shape[0] * 0.96
                if contour_area < image_area * 0.01 or contour_area > image_area * 0.55:
                    continue
                if fills_frame:
                    continue
                selected_contour = contour
                mask_fg = distance_mask
                break
            if selected_contour is not None:
                break

    if selected_contour is None:
        print(f'- {reference_name}: no se encontro una pastilla valida')
        continue

    filled_mask = np.zeros(image_gray.shape, dtype=np.uint8)
    cv2.drawContours(filled_mask, [selected_contour], -1, 255, -1)
    eroded_mask = cv2.erode(filled_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=1)
    inner_edges = cv2.Canny(image_eq, 45, 120)
    inner_edges = cv2.bitwise_and(inner_edges, eroded_mask)

    contour_area = cv2.contourArea(selected_contour)
    contour_perimeter = max(cv2.arcLength(selected_contour, True), 1.0)
    x, y, w, h = cv2.boundingRect(selected_contour)
    aspect_ratio = w / max(h, 1)
    circularity = float(4.0 * np.pi * contour_area / (contour_perimeter * contour_perimeter))
    hull = cv2.convexHull(selected_contour)
    hull_area = max(cv2.contourArea(hull), 1.0)
    solidity = float(contour_area / hull_area)
    hu = cv2.HuMoments(selected_contour).flatten()
    hu = np.sign(hu) * np.log10(np.abs(hu) + 1e-12)

    rotated_rect = cv2.minAreaRect(selected_contour)
    rotated_angle = rotated_rect[2]
    if rotated_rect[1][0] < rotated_rect[1][1]:
        rotated_angle = rotated_angle + 90.0
    rotation_matrix = cv2.getRotationMatrix2D(rotated_rect[0], rotated_angle, 1.0)
    rotated_mask = cv2.warpAffine(filled_mask, rotation_matrix, (image_bgr.shape[1], image_bgr.shape[0]), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    rotated_inner_edges = cv2.warpAffine(inner_edges, rotation_matrix, (image_bgr.shape[1], image_bgr.shape[0]), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    rotated_contours, _ = cv2.findContours(rotated_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    rotated_contours = sorted(rotated_contours, key=cv2.contourArea, reverse=True)
    if rotated_contours:
        rx, ry, rw, rh = cv2.boundingRect(rotated_contours[0])
    else:
        rx, ry, rw, rh = x, y, w, h

    crop_mask = rotated_mask[ry:ry + rh, rx:rx + rw]
    crop_inner_edges = rotated_inner_edges[ry:ry + rh, rx:rx + rw]
    if crop_mask.size == 0:
        crop_mask = filled_mask[y:y + h, x:x + w]
        crop_inner_edges = inner_edges[y:y + h, x:x + w]

    mask_template = cv2.resize(crop_mask, (CANVAS_SIZE, CANVAS_SIZE), interpolation=cv2.INTER_NEAREST).astype(np.float32) / 255.0
    edge_template = cv2.resize(crop_inner_edges, (CANVAS_SIZE, CANVAS_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0
    edge_template = np.clip(edge_template, 0.0, 1.0)
    edge_proj_x = edge_template.mean(axis=0).astype(np.float32)
    edge_proj_y = edge_template.mean(axis=1).astype(np.float32)

    inner_component_count = 0
    inner_components = cv2.connectedComponents((crop_inner_edges > 0).astype(np.uint8))[1]
    if inner_components is not None:
        inner_component_count = max(0, int(inner_components.max()))
    inner_density = float(np.count_nonzero(crop_inner_edges) / max(np.count_nonzero(crop_mask), 1))

    hs_hist = cv2.calcHist([image_hsv], [0, 1], filled_mask, [18, 8], [0, 180, 0, 256])
    cv2.normalize(hs_hist, hs_hist, 0, 1, cv2.NORM_MINMAX)

    pill_pixels_hsv = image_hsv[filled_mask > 0]
    pill_pixels_lab = image_lab[filled_mask > 0]
    mean_lab = pill_pixels_lab.mean(axis=0).astype(np.float32)
    mean_hsv = pill_pixels_hsv.mean(axis=0).astype(np.float32)

    color_analysis = classify_colors_from_mask(image_hsv, filled_mask, min_region_ratio=0.012, min_color_ratio=0.18)
    dominant_colors = color_analysis['dominant_colors']
    dominant_props = color_analysis['dominant_props']
    display_color = color_analysis['display_color']

    reference_profiles[reference_name] = {
        'reference': reference_name,
        'family': item['family'],
        'medicine_name': item['medicine_name'],
        'pillbox_size': item['size_display'],
        'pillbox_mark': item['pillbox_mark'],
        'mark_display': item['mark_display'],
        'pillbox_color': item['pillbox_color'],
        'pillbox_shape': item['pillbox_shape'],
        'image_path': image_path,
        'contour': selected_contour,
        'mask_template': mask_template,
        'edge_template': edge_template,
        'edge_proj_x': edge_proj_x,
        'edge_proj_y': edge_proj_y,
        'hs_hist': hs_hist,
        'mean_lab': mean_lab,
        'mean_hsv': mean_hsv,
        'aspect_ratio': float(aspect_ratio),
        'circularity': float(circularity),
        'solidity': float(solidity),
        'hu': hu.astype(np.float32),
        'inner_density': float(inner_density),
        'inner_component_count': int(inner_component_count),
        'dominant_colors': dominant_colors,
        'dominant_props': dominant_props,
        'display_color': display_color
    }

    family_buckets.setdefault(item['family'], []).append(reference_name)
    print(f"- {reference_name:<22s} | fam={item['family']:<15s} | color={display_color:<13s} | AR={aspect_ratio:>5.2f} | circ={circularity:>5.2f} | sol={solidity:>5.2f}")

print()
print(f'Total de referencias calibradas: {len(reference_profiles)}')

- 60429-203_M_LH3        | fam=ROUND_SOLID     | color=GRAY          | AR= 1.00 | circ= 0.88 | sol= 0.99
- 317220267              | fam=ROUND_SOLID     | color=RED           | AR= 1.01 | circ= 0.71 | sol= 0.96
- 634810623              | fam=ROUND_SOLID     | color=BLUE          | AR= 1.02 | circ= 0.90 | sol= 0.99
- blackNyellow_capsule   | fam=CAPSULE_BICOLOR | color=BLACK/YELLOW  | AR= 2.92 | circ= 0.57 | sol= 0.97
- bloe_oval              | fam=OBLONG_SOLID    | color=BLUE          | AR= 2.44 | circ= 0.63 | sol= 0.98
- brown_oval             | fam=OBLONG_SOLID    | color=BROWN/RED     | AR= 1.61 | circ= 0.71 | sol= 0.97
- cream_capsule          | fam=CAPSULE_BICOLOR | color=GRAY          | AR= 2.70 | circ= 0.62 | sol= 0.98
- green_capsule          | fam=CAPSULE_SOLID   | color=GREEN/BLUE    | AR= 2.63 | circ= 0.59 | sol= 0.97
- green_oval             | fam=OBLONG_SOLID    | color=GRAY          | AR= 1.78 | circ= 0.12 | sol= 0.59
- orangeandblue          | fam=CAPSULE_BICOLOR | color=

## Deteccion en camara

In [9]:
if not reference_profiles:
    print('No hay referencias calibradas. Ejecuta la celda anterior primero.')
else:
    family_weights = {
        'ROUND_SOLID': {'color': 0.40, 'shape': 0.25, 'inner': 0.25, 'template': 0.10},
        'OBLONG_SOLID': {'color': 0.30, 'shape': 0.35, 'inner': 0.15, 'template': 0.20},
        'CAPSULE_SOLID': {'color': 0.35, 'shape': 0.30, 'inner': 0.15, 'template': 0.20},
        'CAPSULE_BICOLOR': {'color': 0.45, 'shape': 0.20, 'inner': 0.15, 'template': 0.20}
    }

    tracks = []
    next_track_id = 1
    frame_number = 0
    previous_mask = None
    bg_model = None
    show_debug = False

    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        print(f'No se pudo abrir la camara {CAMERA_INDEX}. Cambia CAMERA_INDEX en la celda superior.')
    else:
        print('Camara lista. Coloca las pastillas dentro del area de trabajo. q=salir | r=recalibrar fondo | d=debug.')
        print('Segmentacion y matching ajustados para fotos impresas en papel.')
        while True:
            ok, frame_bgr = cap.read()
            if not ok:
                break

            frame_number += 1
            fh, fw = frame_bgr.shape[:2]
            work_w = max(240, int(fw * WORK_AREA_SCALE))
            work_h = max(240, int(fh * WORK_AREA_SCALE))
            wx1 = max(0, (fw - work_w) // 2)
            wy1 = max(0, (fh - work_h) // 2)
            wx2 = min(fw, wx1 + work_w)
            wy2 = min(fh, wy1 + work_h)

            display = frame_bgr.copy()
            shaded = (display.astype(np.float32) * 0.32).astype(np.uint8)
            shaded[wy1:wy2, wx1:wx2] = display[wy1:wy2, wx1:wx2]
            display = shaded

            work_bgr = frame_bgr[wy1:wy2, wx1:wx2].copy()
            work_lab = cv2.cvtColor(work_bgr, cv2.COLOR_BGR2LAB)
            work_hsv = cv2.cvtColor(work_bgr, cv2.COLOR_BGR2HSV)
            work_gray = cv2.cvtColor(work_bgr, cv2.COLOR_BGR2GRAY)
            work_blur = cv2.GaussianBlur(work_gray, (7, 7), 0)
            work_eq = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(work_blur)

            border = max(12, min(work_bgr.shape[:2]) // 16)
            border_lab = np.concatenate([
                work_lab[:border, :, :].reshape(-1, 3),
                work_lab[-border:, :, :].reshape(-1, 3),
                work_lab[:, :border, :].reshape(-1, 3),
                work_lab[:, -border:, :].reshape(-1, 3)
            ], axis=0).astype(np.float32)
            border_hsv = np.concatenate([
                work_hsv[:border, :, :].reshape(-1, 3),
                work_hsv[-border:, :, :].reshape(-1, 3),
                work_hsv[:, :border, :].reshape(-1, 3),
                work_hsv[:, -border:, :].reshape(-1, 3)
            ], axis=0).astype(np.float32)

            if bg_model is None:
                bg_lab = np.median(border_lab, axis=0)
                bg_lab_std = np.std(border_lab, axis=0)
                bg_sat = float(np.median(border_hsv[:, 1]))
                bg_val = float(np.median(border_hsv[:, 2]))
                bg_sat_std = float(np.std(border_hsv[:, 1]))
                bg_chroma = np.sqrt((border_lab[:, 1] - 128.0) ** 2 + (border_lab[:, 2] - 128.0) ** 2)
                bg_chroma_med = float(np.median(bg_chroma))
                bg_model = {
                    'bg_lab': bg_lab,
                    'bg_lab_std': bg_lab_std,
                    'bg_sat': bg_sat,
                    'bg_val': bg_val,
                    'bg_sat_std': bg_sat_std,
                    'bg_chroma_med': bg_chroma_med,
                }
                previous_mask = None
                tracks = []
            else:
                bg_lab = bg_model['bg_lab']
                bg_lab_std = bg_model['bg_lab_std']
                bg_sat = bg_model['bg_sat']
                bg_val = bg_model['bg_val']
                bg_sat_std = bg_model['bg_sat_std']
                bg_chroma_med = bg_model['bg_chroma_med']

            lab_delta = np.abs(work_lab.astype(np.float32) - bg_lab.reshape(1, 1, 3))
            mask_lab = ((lab_delta[:, :, 0] > max(10.0, bg_lab_std[0] * 2.4)) | (lab_delta[:, :, 1] > max(12.0, bg_lab_std[1] * 2.2)) | (lab_delta[:, :, 2] > max(12.0, bg_lab_std[2] * 2.2))).astype(np.uint8) * 255
            sat_delta = np.abs(work_hsv[:, :, 1].astype(np.float32) - bg_sat)
            val_delta = np.abs(work_hsv[:, :, 2].astype(np.float32) - bg_val)
            work_chroma = np.sqrt((work_lab[:, :, 1].astype(np.float32) - 128.0) ** 2 + (work_lab[:, :, 2].astype(np.float32) - 128.0) ** 2)
            chroma_delta = np.abs(work_chroma - bg_chroma_med)
            neutral_background = ((work_hsv[:, :, 1].astype(np.float32) < max(NEUTRAL_SAT_LIMIT, bg_sat + 10.0)) & (work_chroma < max(NEUTRAL_CHROMA_LIMIT, bg_chroma_med + 6.0)) & (lab_delta[:, :, 0] < max(13.0, bg_lab_std[0] * 2.0)) & (chroma_delta < max(10.0, bg_chroma_med * 0.55 + 4.0))).astype(np.uint8) * 255
            mask_sv = ((sat_delta > max(22.0, bg_sat_std * 2.2)) | (val_delta > 30.0)).astype(np.uint8) * 255
            relaxed_color_gate = ((sat_delta > max(10.0, bg_sat_std * 1.2)) | (val_delta > 14.0) | (chroma_delta > max(6.0, bg_chroma_med * 0.35 + 2.5))).astype(np.uint8) * 255
            color_support_mask = build_color_support_mask(work_hsv)
            color_support_mask = cv2.bitwise_and(color_support_mask, relaxed_color_gate)
            local_bg = cv2.GaussianBlur(work_eq, (31, 31), 0)
            local_contrast = np.abs(work_eq.astype(np.float32) - local_bg.astype(np.float32))
            print_ink_mask = ((work_hsv[:, :, 1].astype(np.float32) > PRINT_MIN_SAT) | (chroma_delta > max(PRINT_CHROMA_DELTA, bg_chroma_med * 0.20 + 1.5)) | (local_contrast > PRINT_LOCAL_CONTRAST)).astype(np.uint8) * 255
            print_ink_mask = cv2.morphologyEx(print_ink_mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)), iterations=1)
            print_ink_mask = cv2.morphologyEx(print_ink_mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)), iterations=1)
            current_mask = cv2.max(cv2.max(mask_lab, mask_sv), color_support_mask)
            current_mask = cv2.max(current_mask, print_ink_mask)
            support_union = cv2.bitwise_or(color_support_mask, print_ink_mask)
            neutral_block = cv2.bitwise_and(neutral_background, cv2.bitwise_not(support_union))
            current_mask = cv2.bitwise_and(current_mask, cv2.bitwise_not(neutral_block))
            current_mask = cv2.GaussianBlur(current_mask, (7, 7), 0)
            _, current_mask = cv2.threshold(current_mask, 104, 255, cv2.THRESH_BINARY)
            current_mask = cv2.morphologyEx(current_mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17)), iterations=2)
            current_mask = cv2.morphologyEx(current_mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ROUND_CLOSE_KERNEL, ROUND_CLOSE_KERNEL)), iterations=1)
            current_mask = cv2.morphologyEx(current_mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)), iterations=1)
            current_mask = cv2.dilate(current_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=1)
            current_mask = cv2.medianBlur(current_mask, 9)

            stable_fg_mask = current_mask.copy()
            if previous_mask is not None and previous_mask.shape == current_mask.shape:
                previous_support = cv2.dilate(previous_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)), iterations=1)
                stable_core = cv2.bitwise_and(current_mask, previous_mask)
                stable_support = cv2.bitwise_and(current_mask, previous_support)
                stable_fg_mask = cv2.addWeighted(stable_support, 1.0 - MASK_MEMORY, stable_core, MASK_MEMORY, 0.0)
                _, stable_fg_mask = cv2.threshold(stable_fg_mask, 96, 255, cv2.THRESH_BINARY)
                if np.count_nonzero(stable_fg_mask) < max(500, int(np.count_nonzero(current_mask) * 0.45)):
                    stable_fg_mask = current_mask.copy()

            stable_fg_mask = cv2.morphologyEx(stable_fg_mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (19, 19)), iterations=2)
            stable_fg_mask = cv2.morphologyEx(stable_fg_mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)), iterations=1)
            stable_fg_mask = cv2.medianBlur(stable_fg_mask, 9)
            previous_mask = stable_fg_mask.copy()

            work_inner_edges = cv2.Canny(work_eq, 60, 150)
            work_inner_edges = cv2.bitwise_and(work_inner_edges, cv2.erode(stable_fg_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=1))
            work_inner_edges = cv2.morphologyEx(work_inner_edges, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)), iterations=1)

            display_roi = display[wy1:wy2, wx1:wx2]
            if show_debug:
                display_roi[work_inner_edges > 0] = (255, 255, 0)

            contours, _ = cv2.findContours(stable_fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            contours = sorted(contours, key=cv2.contourArea, reverse=True)
            detections = []
            work_area = float(work_bgr.shape[0] * work_bgr.shape[1])
            frame_match_accept = max(0.0, MATCH_ACCEPT - PRINT_ACCEPT_DELTA)

            for contour in contours:
                if len(detections) >= MAX_DETECTIONS:
                    break
                raw_area = cv2.contourArea(contour)
                if raw_area < MIN_COMPONENT_AREA:
                    continue
                if raw_area > work_area * MAX_COMPONENT_AREA_RATIO:
                    continue

                stabilized_contour = contour.copy()
                contour_area = cv2.contourArea(stabilized_contour)
                x, y, w, h = cv2.boundingRect(stabilized_contour)
                if w < 35 or h < 35:
                    continue
                round_hint = abs(w - h) / max(float(max(w, h)), 1.0) < 0.22
                fill_ratio = contour_area / max(float(w * h), 1.0)
                if fill_ratio < 0.50:
                    continue

                candidate_mask = np.zeros(work_gray.shape, dtype=np.uint8)
                cv2.drawContours(candidate_mask, [stabilized_contour], -1, 255, -1)
                candidate_mask = cv2.morphologyEx(candidate_mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)), iterations=2)
                candidate_mask = cv2.morphologyEx(candidate_mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=1)
                if round_hint:
                    candidate_mask = cv2.morphologyEx(candidate_mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15)), iterations=2)
                    candidate_mask = cv2.medianBlur(candidate_mask, 7)
                candidate_contours, _ = cv2.findContours(candidate_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                if not candidate_contours:
                    continue
                stabilized_contour = max(candidate_contours, key=cv2.contourArea)
                contour_area = cv2.contourArea(stabilized_contour)
                x, y, w, h = cv2.boundingRect(stabilized_contour)
                eroded_candidate = cv2.erode(candidate_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=1)
                candidate_inner_edges = cv2.bitwise_and(work_inner_edges, eroded_candidate)

                perimeter = max(cv2.arcLength(stabilized_contour, True), 1.0)
                aspect_ratio = w / max(h, 1)
                circularity = float(4.0 * np.pi * contour_area / (perimeter * perimeter))
                hull = cv2.convexHull(stabilized_contour)
                hull_area = max(cv2.contourArea(hull), 1.0)
                solidity = float(contour_area / hull_area)
                if solidity < 0.72:
                    continue
                if round_hint and aspect_ratio <= 1.20 and circularity < 0.80 and solidity >= 0.70:
                    circle_center, circle_radius = cv2.minEnclosingCircle(stabilized_contour)
                    if circle_radius >= 18.0:
                        circle_mask = np.zeros(work_gray.shape, dtype=np.uint8)
                        cv2.circle(circle_mask, (int(round(circle_center[0])), int(round(circle_center[1]))), int(round(circle_radius * 0.96)), 255, -1)
                        overlap_ratio = float(np.count_nonzero(cv2.bitwise_and(circle_mask, candidate_mask)) / max(np.count_nonzero(candidate_mask), 1))
                        if overlap_ratio >= 0.74:
                            candidate_mask = cv2.bitwise_or(candidate_mask, circle_mask)
                            candidate_mask = cv2.morphologyEx(candidate_mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15)), iterations=1)
                            candidate_mask = cv2.medianBlur(candidate_mask, 7)
                            candidate_contours, _ = cv2.findContours(candidate_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                            if candidate_contours:
                                stabilized_contour = max(candidate_contours, key=cv2.contourArea)
                                contour_area = cv2.contourArea(stabilized_contour)
                                x, y, w, h = cv2.boundingRect(stabilized_contour)
                                perimeter = max(cv2.arcLength(stabilized_contour, True), 1.0)
                                aspect_ratio = w / max(h, 1)
                                circularity = float(4.0 * np.pi * contour_area / (perimeter * perimeter))
                                hull = cv2.convexHull(stabilized_contour)
                                hull_area = max(cv2.contourArea(hull), 1.0)
                                solidity = float(contour_area / hull_area)

                hu = cv2.HuMoments(stabilized_contour).flatten()
                hu = np.sign(hu) * np.log10(np.abs(hu) + 1e-12)

                rotated_rect = cv2.minAreaRect(stabilized_contour)
                rotated_angle = rotated_rect[2]
                if rotated_rect[1][0] < rotated_rect[1][1]:
                    rotated_angle = rotated_angle + 90.0
                rotation_matrix = cv2.getRotationMatrix2D(rotated_rect[0], rotated_angle, 1.0)
                rotated_mask = cv2.warpAffine(candidate_mask, rotation_matrix, (work_bgr.shape[1], work_bgr.shape[0]), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
                rotated_inner = cv2.warpAffine(candidate_inner_edges, rotation_matrix, (work_bgr.shape[1], work_bgr.shape[0]), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
                rotated_contours, _ = cv2.findContours(rotated_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                rotated_contours = sorted(rotated_contours, key=cv2.contourArea, reverse=True)
                if rotated_contours:
                    rx, ry, rw, rh = cv2.boundingRect(rotated_contours[0])
                else:
                    rx, ry, rw, rh = x, y, w, h

                crop_mask = rotated_mask[ry:ry + rh, rx:rx + rw]
                crop_inner_edges = rotated_inner[ry:ry + rh, rx:rx + rw]
                if crop_mask.size == 0:
                    crop_mask = candidate_mask[y:y + h, x:x + w]
                    crop_inner_edges = candidate_inner_edges[y:y + h, x:x + w]

                mask_template = cv2.resize(crop_mask, (CANVAS_SIZE, CANVAS_SIZE), interpolation=cv2.INTER_NEAREST).astype(np.float32) / 255.0
                edge_template = cv2.resize(crop_inner_edges, (CANVAS_SIZE, CANVAS_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0
                edge_template = np.clip(edge_template, 0.0, 1.0)
                edge_proj_x = edge_template.mean(axis=0).astype(np.float32)
                edge_proj_y = edge_template.mean(axis=1).astype(np.float32)
                inner_component_labels = cv2.connectedComponents((crop_inner_edges > 0).astype(np.uint8))[1]
                inner_component_count = max(0, int(inner_component_labels.max())) if inner_component_labels is not None else 0
                inner_density = float(np.count_nonzero(crop_inner_edges) / max(np.count_nonzero(crop_mask), 1))

                candidate_pixels = candidate_mask > 0
                background_like_pixels = int(np.count_nonzero(neutral_background[candidate_pixels] > 0))
                candidate_pixels_count = max(int(np.count_nonzero(candidate_pixels)), 1)
                background_like_ratio = float(background_like_pixels / candidate_pixels_count)
                if background_like_ratio >= BACKGROUND_REJECT_RATIO:
                    continue

                pill_pixels_hsv = work_hsv[candidate_pixels]
                pill_pixels_lab = work_lab[candidate_pixels]
                if pill_pixels_hsv.size == 0:
                    continue

                hs_hist = cv2.calcHist([work_hsv], [0, 1], candidate_mask, [18, 8], [0, 180, 0, 256])
                cv2.normalize(hs_hist, hs_hist, 0, 1, cv2.NORM_MINMAX)
                mean_lab = pill_pixels_lab.mean(axis=0).astype(np.float32)
                mean_hsv = pill_pixels_hsv.mean(axis=0).astype(np.float32)

                candidate_min_color_ratio = PRINT_BICOLOR_MIN_RATIO
                color_analysis = classify_colors_from_mask(work_hsv, candidate_mask, min_region_ratio=0.015, min_color_ratio=candidate_min_color_ratio)
                dominant_colors = color_analysis['dominant_colors']
                dominant_props = color_analysis['dominant_props']
                is_bicolor = color_analysis['is_bicolor']
                display_color = color_analysis['display_color']

                if aspect_ratio <= 1.15 and circularity >= 0.78:
                    shape_label = 'ROUND'
                    candidate_family = 'ROUND_SOLID'
                    comparison_pool = family_buckets.get('ROUND_SOLID', [])
                    weights = family_weights['ROUND_SOLID']
                elif is_bicolor:
                    shape_label = 'CAPSULE' if aspect_ratio >= 1.45 else 'OVAL'
                    candidate_family = 'CAPSULE_BICOLOR'
                    comparison_pool = family_buckets.get('CAPSULE_BICOLOR', [])
                    weights = family_weights['CAPSULE_BICOLOR']
                else:
                    shape_label = 'CAPSULE' if aspect_ratio >= 1.70 else 'OVAL'
                    candidate_family = 'CAPSULE_SOLID' if aspect_ratio >= 1.95 else 'OBLONG_SOLID'
                    comparison_pool = family_buckets.get('OBLONG_SOLID', []) + family_buckets.get('CAPSULE_SOLID', [])
                    weights = family_weights['CAPSULE_SOLID'] if aspect_ratio >= 1.95 else family_weights['OBLONG_SOLID']

                match_rows = []
                for reference_name in comparison_pool:
                    profile = reference_profiles.get(reference_name)
                    if profile is None:
                        continue

                    hist_corr = float(cv2.compareHist(hs_hist, profile['hs_hist'], cv2.HISTCMP_CORREL))
                    hist_corr = max(0.0, min(1.0, (hist_corr + 1.0) / 2.0))
                    mean_lab_distance = float(np.linalg.norm(mean_lab - profile['mean_lab']))
                    mean_lab_score = 1.0 - min(mean_lab_distance / 90.0, 1.0)

                    color_name_score = 0.0
                    if dominant_colors[0] == profile['dominant_colors'][0]:
                        color_name_score += 0.55
                    elif dominant_colors[0] in profile['dominant_colors'][:2]:
                        color_name_score += 0.35
                    if len(dominant_colors) > 1 and len(profile['dominant_colors']) > 1 and dominant_colors[1] == profile['dominant_colors'][1]:
                        color_name_score += 0.25
                    elif len(dominant_colors) > 1 and dominant_colors[1] in profile['dominant_colors'][:2]:
                        color_name_score += 0.12
                    color_name_score = min(color_name_score, 1.0)

                    profile_secondary = profile['dominant_props'][1] if len(profile['dominant_props']) > 1 else 0.0
                    candidate_secondary = dominant_props[1] if len(dominant_props) > 1 else 0.0
                    split_score = 1.0 - min(abs(candidate_secondary - profile_secondary) / 0.45, 1.0)
                    candidate_sat_median = float(np.median(pill_pixels_hsv[:, 1]))
                    if candidate_sat_median <= PRINT_LOW_SAT_THRESHOLD:
                        score_color = 0.30 * hist_corr + 0.42 * mean_lab_score + 0.08 * color_name_score + 0.20 * split_score
                    else:
                        score_color = 0.45 * hist_corr + 0.30 * mean_lab_score + 0.15 * color_name_score + 0.10 * split_score

                    ar_score = 1.0 - min(abs(aspect_ratio - profile['aspect_ratio']) / max(profile['aspect_ratio'], 0.20), 1.0)
                    circ_score = 1.0 - min(abs(circularity - profile['circularity']) / max(profile['circularity'], 0.20), 1.0)
                    sol_score = 1.0 - min(abs(solidity - profile['solidity']) / max(profile['solidity'], 0.20), 1.0)
                    hu_distance = float(np.mean(np.abs(hu - profile['hu'])))
                    hu_score = 1.0 - min(hu_distance / 5.5, 1.0)
                    match_shapes_distance = float(cv2.matchShapes(stabilized_contour, profile['contour'], cv2.CONTOURS_MATCH_I1, 0.0))
                    match_shapes_score = 1.0 - min(match_shapes_distance / 0.70, 1.0)
                    score_shape = 0.24 * ar_score + 0.24 * circ_score + 0.16 * sol_score + 0.18 * hu_score + 0.18 * match_shapes_score

                    density_score = 1.0 - min(abs(inner_density - profile['inner_density']) / max(profile['inner_density'], 0.04), 1.0)
                    component_score = 1.0 - min(abs(inner_component_count - profile['inner_component_count']) / max(profile['inner_component_count'] + 1, 2), 1.0)
                    proj_x_den = float(np.linalg.norm(edge_proj_x) * np.linalg.norm(profile['edge_proj_x']))
                    proj_y_den = float(np.linalg.norm(edge_proj_y) * np.linalg.norm(profile['edge_proj_y']))
                    proj_x_score = float(np.dot(edge_proj_x, profile['edge_proj_x']) / proj_x_den) if proj_x_den > 1e-6 else 0.0
                    proj_y_score = float(np.dot(edge_proj_y, profile['edge_proj_y']) / proj_y_den) if proj_y_den > 1e-6 else 0.0
                    proj_x_score = max(0.0, min(1.0, proj_x_score))
                    proj_y_score = max(0.0, min(1.0, proj_y_score))
                    score_inner = 0.35 * density_score + 0.15 * component_score + 0.25 * proj_x_score + 0.25 * proj_y_score

                    mask_template_score = 1.0 - min(float(np.mean(np.abs(mask_template - profile['mask_template']))), 1.0)
                    edge_template_score = 1.0 - min(float(np.mean(np.abs(edge_template - profile['edge_template']))) * 1.8, 1.0)
                    score_template = 0.60 * mask_template_score + 0.40 * edge_template_score

                    total_score = weights['color'] * score_color + weights['shape'] * score_shape + weights['inner'] * score_inner + weights['template'] * score_template
                    if candidate_family == 'ROUND_SOLID' and profile['family'] != 'ROUND_SOLID':
                        total_score = total_score * 0.70
                    if candidate_family == 'CAPSULE_BICOLOR' and profile['family'] != 'CAPSULE_BICOLOR':
                        total_score = total_score * 0.70
                    if candidate_family != 'ROUND_SOLID' and candidate_family != 'CAPSULE_BICOLOR':
                        if candidate_family == 'CAPSULE_SOLID' and profile['family'] == 'OBLONG_SOLID':
                            total_score = total_score * 0.93
                        if candidate_family == 'OBLONG_SOLID' and profile['family'] == 'CAPSULE_SOLID':
                            total_score = total_score * 0.93

                    match_rows.append({
                        'reference': reference_name,
                        'profile': profile,
                        'score_color': float(score_color),
                        'score_shape': float(score_shape),
                        'score_inner': float(score_inner),
                        'score_template': float(score_template),
                        'score_total': float(total_score)
                    })

                match_rows = sorted(match_rows, key=lambda row: row['score_total'], reverse=True)
                top1 = match_rows[0] if match_rows else None
                top2 = match_rows[1] if len(match_rows) > 1 else None
                accepted = False
                if top1 is not None:
                    if top2 is None:
                        accepted = top1['score_total'] >= frame_match_accept
                    else:
                        accepted = top1['score_total'] >= frame_match_accept and (top1['score_total'] - top2['score_total']) >= MATCH_MARGIN
                    if top1['score_shape'] < 0.52 or top1['score_template'] < 0.48:
                        accepted = False

                center_x = wx1 + x + w / 2.0
                center_y = wy1 + y + h / 2.0
                global_bbox = (wx1 + x, wy1 + y, w, h)
                global_contour = stabilized_contour.copy()
                global_contour[:, 0, 0] += wx1
                global_contour[:, 0, 1] += wy1
                raw_label = top1['reference'] if accepted and top1 is not None else 'INDETERMINADO'

                matched_track = None
                matched_distance = TRACK_RADIUS + 1.0
                for track in tracks:
                    if frame_number - track['last_seen'] > 8:
                        continue
                    dx = center_x - track['center'][0]
                    dy = center_y - track['center'][1]
                    distance = float(np.hypot(dx, dy))
                    if distance < matched_distance and distance <= TRACK_RADIUS:
                        matched_track = track
                        matched_distance = distance

                snapshot = {
                    'label': raw_label,
                    'accepted': accepted,
                    'reference': top1['reference'] if top1 is not None else '',
                    'score_color': top1['score_color'] if top1 is not None else 0.0,
                    'score_shape': top1['score_shape'] if top1 is not None else 0.0,
                    'score_inner': top1['score_inner'] if top1 is not None else 0.0,
                    'score_template': top1['score_template'] if top1 is not None else 0.0,
                    'score_total': top1['score_total'] if top1 is not None else 0.0
                }

                if matched_track is None:
                    matched_track = {
                        'id': next_track_id,
                        'center': (center_x, center_y),
                        'bbox': global_bbox,
                        'last_seen': frame_number,
                        'history': [snapshot]
                    }
                    next_track_id += 1
                    tracks.append(matched_track)
                else:
                    prev_center_x, prev_center_y = matched_track['center']
                    prev_x, prev_y, prev_w, prev_h = matched_track['bbox']
                    matched_track['center'] = (
                        BOX_BLEND * prev_center_x + (1.0 - BOX_BLEND) * center_x,
                        BOX_BLEND * prev_center_y + (1.0 - BOX_BLEND) * center_y
                    )
                    matched_track['bbox'] = (
                        int(round(BOX_BLEND * prev_x + (1.0 - BOX_BLEND) * global_bbox[0])),
                        int(round(BOX_BLEND * prev_y + (1.0 - BOX_BLEND) * global_bbox[1])),
                        int(round(BOX_BLEND * prev_w + (1.0 - BOX_BLEND) * global_bbox[2])),
                        int(round(BOX_BLEND * prev_h + (1.0 - BOX_BLEND) * global_bbox[3]))
                    )
                    matched_track['last_seen'] = frame_number
                    matched_track['history'].append(snapshot)
                    matched_track['history'] = matched_track['history'][-TRACK_MEMORY:]

                recent_history = matched_track['history'][-TRACK_MEMORY:]
                accepted_labels = [row['label'] for row in recent_history if row['accepted'] and row['label'] != 'INDETERMINADO']
                display_reference = top1['reference'] if top1 is not None else ''
                display_color_score = top1['score_color'] if top1 is not None else 0.0
                display_shape_score = top1['score_shape'] if top1 is not None else 0.0
                display_inner_score = top1['score_inner'] if top1 is not None else 0.0
                display_template_score = top1['score_template'] if top1 is not None else 0.0
                display_total_score = top1['score_total'] if top1 is not None else 0.0
                display_label = 'INDETERMINADO'

                if len(accepted_labels) >= 2:
                    label_counts = {}
                    for label in accepted_labels:
                        label_counts[label] = label_counts.get(label, 0) + 1
                    majority_label = max(label_counts, key=label_counts.get)
                    if label_counts[majority_label] >= 2:
                        display_label = majority_label
                        stable_rows = [row for row in recent_history if row['label'] == majority_label and row['accepted']]
                        display_reference = majority_label
                        display_color_score = float(np.mean([row['score_color'] for row in stable_rows]))
                        display_shape_score = float(np.mean([row['score_shape'] for row in stable_rows]))
                        display_inner_score = float(np.mean([row['score_inner'] for row in stable_rows]))
                        display_template_score = float(np.mean([row['score_template'] for row in stable_rows]))
                        display_total_score = float(np.mean([row['score_total'] for row in stable_rows]))
                if display_label == 'INDETERMINADO' and accepted and top1 is not None and top1['score_total'] >= 0.80:
                    display_label = top1['reference']

                detections.append({
                    'bbox': matched_track['bbox'],
                    'contour': global_contour,
                    'shape_label': shape_label,
                    'display_color': display_color,
                    'candidate_family': candidate_family,
                    'top1': top1,
                    'top2': top2,
                    'display_label': display_label,
                    'display_reference': display_reference,
                    'display_color_score': display_color_score,
                    'display_shape_score': display_shape_score,
                    'display_inner_score': display_inner_score,
                    'display_template_score': display_template_score,
                    'display_total_score': display_total_score
                })

            tracks = [track for track in tracks if frame_number - track['last_seen'] <= 8]

            cv2.rectangle(display, (wx1, wy1), (wx2, wy2), (80, 180, 255), 2)
            cv2.putText(display, 'Area de trabajo', (wx1 + 8, max(22, wy1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.60, (80, 180, 255), 2, cv2.LINE_AA)

            accepted_count = 0
            for det in detections:
                x, y, w, h = det['bbox']
                contour = det['contour']
                top1 = det['top1']
                top2 = det['top2']
                display_label = det['display_label']
                shape_label = det['shape_label']
                display_color = det['display_color']
                candidate_family = det['candidate_family']
                score_color = det['display_color_score']
                score_shape = det['display_shape_score']
                score_inner = det['display_inner_score']
                score_template = det['display_template_score']
                score_total = det['display_total_score']

                contour_color = (0, 255, 0) if display_label != 'INDETERMINADO' else (0, 210, 255)
                cv2.drawContours(display, [contour], -1, contour_color, 2)
                cv2.rectangle(display, (x, y), (x + w, y + h), contour_color, 2)

                line_1 = f'{display_label} | {display_color} | {shape_label}'
                if display_label != 'INDETERMINADO' and display_label in reference_profiles:
                    accepted_count += 1
                    ref_profile = reference_profiles[display_label]
                    line_2 = f"size={ref_profile['pillbox_size']} | mark={ref_profile['mark_display']}"
                else:
                    best_name = top1['reference'] if top1 is not None else 'NONE'
                    gap = (top1['score_total'] - top2['score_total']) if (top1 is not None and top2 is not None) else 0.0
                    line_2 = f'best={best_name} | gap={gap:.2f} | fam={candidate_family}'
                line_3 = f'C={score_color:.2f} F={score_shape:.2f} BI={score_inner:.2f} T={score_template:.2f} S={score_total:.2f}'

                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 0.46
                font_thickness = 1
                text_lines = [line_1, line_2, line_3]
                text_widths = [cv2.getTextSize(text, font, font_scale, font_thickness)[0][0] for text in text_lines]
                block_width = min(fw - x - 1, max(text_widths) + 12)
                block_height = 18 * len(text_lines) + 8
                block_y = max(0, y - block_height - 4)
                block_x2 = min(fw - 1, x + block_width)
                block_y2 = min(fh - 1, block_y + block_height)
                cv2.rectangle(display, (x, block_y), (block_x2, block_y2), (15, 15, 15), -1)
                cv2.rectangle(display, (x, block_y), (block_x2, block_y2), contour_color, 1)
                for idx, text in enumerate(text_lines):
                    text_y = block_y + 16 + idx * 18
                    cv2.putText(display, text, (x + 6, text_y), font, font_scale, (255, 255, 255), font_thickness, cv2.LINE_AA)

            summary = f'area={wx2 - wx1}x{wy2 - wy1} | candidatos={len(detections)}/{MAX_DETECTIONS} | aceptadas={accepted_count} | umbral={frame_match_accept:.2f} | margen={MATCH_MARGIN:.2f}'
            cv2.putText(display, summary, (14, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(display, 'Dentro del recuadro. q=salir | r=recalibrar fondo | d=debug | verde=match | amarillo=indeterminado', (14, display.shape[0] - 16), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (230, 230, 230), 1, cv2.LINE_AA)

            cv2.imshow('Detector de Pastillas - Decision por familias', display)
            if show_debug:
                debug_mask = cv2.cvtColor(stable_fg_mask, cv2.COLOR_GRAY2BGR)
                debug_current = cv2.cvtColor(current_mask, cv2.COLOR_GRAY2BGR)
                debug_ink = cv2.cvtColor(print_ink_mask, cv2.COLOR_GRAY2BGR)
                debug_edges = cv2.cvtColor(work_inner_edges, cv2.COLOR_GRAY2BGR)
                cv2.putText(debug_current, 'mask actual', (10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (0, 255, 255), 2, cv2.LINE_AA)
                cv2.putText(debug_mask, 'mask estable', (10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (0, 255, 255), 2, cv2.LINE_AA)
                cv2.putText(debug_ink, 'ink print', (10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (0, 255, 255), 2, cv2.LINE_AA)
                cv2.putText(debug_edges, 'bordes internos', (10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (0, 255, 255), 2, cv2.LINE_AA)
                debug_panel = np.hstack([debug_current, debug_mask, debug_ink, debug_edges])
                cv2.imshow('Detector Debug', debug_panel)
            else:
                try:
                    cv2.destroyWindow('Detector Debug')
                except cv2.error:
                    pass

            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            if key == ord('d'):
                show_debug = not show_debug
            if key == ord('r'):
                bg_model = None
                previous_mask = None
                tracks = []
                print('Recalibrando fondo con el siguiente frame...')

        cap.release()
        cv2.destroyAllWindows()

Camara lista. Coloca las pastillas dentro del area de trabajo. q=salir | r=recalibrar fondo | d=debug.
Segmentacion y matching ajustados para fotos impresas en papel.
